In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
#normalize scores
def normalize(arr):
    return (arr - arr.min()) / (np.ptp(arr) + 1e-8)


In [4]:
#building user text vector
def build_user_text_vector(user_id, ratings_df, text_embeddings, movie_id_to_index):
    seen = ratings_df[ratings_df.userId == user_id]["movieId"]

    indices = [
        movie_id_to_index[mid]
        for mid in seen if mid in movie_id_to_index
    ]

    if len(indices) == 0:
        return None

    return text_embeddings[indices].mean(axis=0)


In [5]:
#building user poster vector

def build_user_poster_vector(user_id, ratings_df, poster_embeddings, movie_id_to_index):
    seen = ratings_df[ratings_df.userId == user_id]["movieId"]

    indices = [
        movie_id_to_index[mid]
        for mid in seen
        if mid in movie_id_to_index and
           np.linalg.norm(poster_embeddings[movie_id_to_index[mid]]) > 0
    ]

    if len(indices) == 0:
        return None

    return poster_embeddings[indices].mean(axis=0)


In [6]:
#Multimodal hybrid recommender

def recommend_movies_multimodal(
    user_id,
    ncf_model,
    ratings_df,
    movies_df,
    text_embeddings,
    poster_embeddings,
    user_encoder,
    movie_encoder,
    alpha=0.5,   # NCF
    beta=0.3,    # Text
    gamma=0.2,   # Poster
    top_n=10
):
    assert abs(alpha + beta + gamma - 1.0) < 1e-6

    # ---------------- Encode user ----------------
    user_encoded = user_encoder.transform([user_id])[0]

    seen_movies = set(
        ratings_df[ratings_df.userId == user_id]["movieId"]
    )

    candidate_movies = [
        m for m in movies_df.movieId
        if m not in seen_movies and m in movie_encoder.classes_
    ]

    candidate_encoded = movie_encoder.transform(candidate_movies)

    # ---------------- NCF ----------------
    user_input = np.full(len(candidate_encoded), user_encoded)

    ncf_scores = ncf_model.predict(
        [user_input, candidate_encoded],
        batch_size=1024,
        verbose=0
    ).flatten()

    ncf_scores = normalize(ncf_scores)

    # ---------------- TEXT ----------------
    movie_id_to_index = dict(zip(movies_df.movieId, movies_df.index))

    user_text_vec = build_user_text_vector(
        user_id, ratings_df, text_embeddings, movie_id_to_index
    )

    if user_text_vec is not None:
        candidate_indices = [movie_id_to_index[m] for m in candidate_movies]
        text_scores = cosine_similarity(
            user_text_vec.reshape(1, -1),
            text_embeddings[candidate_indices]
        )[0]
        text_scores = normalize(text_scores)
    else:
        text_scores = np.zeros(len(candidate_movies))

    # ---------------- POSTER ----------------
    user_poster_vec = build_user_poster_vector(
        user_id, ratings_df, poster_embeddings, movie_id_to_index
    )

    if user_poster_vec is not None:
        poster_scores_raw = cosine_similarity(
            user_poster_vec.reshape(1, -1),
            poster_embeddings[candidate_indices]
        )[0]

        valid_mask = np.linalg.norm(
            poster_embeddings[candidate_indices], axis=1
        ) > 0

        poster_scores = np.zeros(len(candidate_movies))
        poster_scores[valid_mask] = normalize(poster_scores_raw[valid_mask])
    else:
        poster_scores = np.zeros(len(candidate_movies))


        # ---------------- FINAL FUSION ----------------
    final_scores = (
        alpha * ncf_scores +
        beta * text_scores +
        gamma * poster_scores
    )

    top_idx = np.argsort(final_scores)[::-1][:top_n]
    rec_ids = [candidate_movies[i] for i in top_idx]

    recs = movies_df[
        movies_df.movieId.isin(rec_ids)
    ][["movieId", "title", "genres"]]

    return recs, ncf_scores, text_scores, poster_scores


In [7]:
from tensorflow.keras.models import load_model

NCF_MODEL_PATH = r"D:\MINI PROJECT\NOTEBOOKS\ncfmodel.keras"
ncfmodel = load_model(NCF_MODEL_PATH)

print("NCF model loaded")


NCF model loaded


In [8]:
ncfmodel.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 50)     │  6,924,650 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 50)     │  1,337,200 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 50)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 50)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 100)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     12,928 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         65 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,849,299 (94.79 MB)

 Trainable params: 8,283,099 (31.60 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 16,566,200 (63.20 MB)

In [ ]:
NCF_MODEL_PATH = r"D:\MINI PROJECT\Checkpoints\ncfmodel.keras"
USER_ENCODER_PATH = r"D:\MINI PROJECT\Checkpoints\user_encoder.pkl"
MOVIE_ENCODER_PATH = r"D:\MINI PROJECT\Checkpoints\movie_encoder.pkl"
EMBEDDINGS_PATH = r"D:\MINI PROJECT\Checkpoints\movie_embeddings.npy"

MOVIES_PATH = r"D:\MINI PROJECT\DATASET\movies_final.csv"
RATINGS_PATH = r"D:\MINI PROJECT\DATASET\MovieLensDataset20M\rating.csv"

In [10]:
import joblib

In [11]:
print("Loading data...")
movies = pd.read_csv(MOVIES_PATH)
ratings = pd.read_csv(RATINGS_PATH)

# -------------------------------
# LOAD MODEL & ENCODERS
# -------------------------------
print("Loading NCF model...")
ncfmodel = load_model(NCF_MODEL_PATH)

print("Loading encoders...")
user_encoder = joblib.load(USER_ENCODER_PATH)
movie_encoder = joblib.load(MOVIE_ENCODER_PATH)


Loading data...
Loading NCF model...
Loading encoders...


In [12]:
import numpy as np

TEXT_EMB_PATH = r"D:\MINI PROJECT\movie_embeddings.npy"
text_embeddings = np.load(TEXT_EMB_PATH)

print(text_embeddings.shape)


(23138, 384)


In [13]:
import numpy as np

TEXT_EMB_PATH = r"D:\MINI PROJECT\poster_embeddings.npy"
poster_embeddings = np.load(TEXT_EMB_PATH)

print(poster_embeddings.shape)

(23138, 2048)


In [14]:
multimodal_recs, ncf_scores, text_scores, poster_scores = recommend_movies_multimodal(
    user_id=11,
    ncf_model=ncfmodel,
    ratings_df=ratings,
    movies_df=movies,
    text_embeddings=text_embeddings,
    poster_embeddings=poster_embeddings,
    user_encoder=user_encoder,
    movie_encoder=movie_encoder,
    alpha=0.5,
    beta=0.3,
    gamma=0.2,
    top_n=10
)


In [15]:
#score contribution
print("NCF mean:", ncf_scores.mean())
print("Text mean:", text_scores.mean())
print("Poster mean:", poster_scores.mean())


NCF mean: 0.74011624
Text mean: 0.59714144
Poster mean: 0.0


In [16]:
movie_id_to_index = dict(zip(movies.movieId, movies.index))

user_poster_vec = build_user_poster_vector(
    user_id=10,
    ratings_df=ratings,
    poster_embeddings=poster_embeddings,
    movie_id_to_index=movie_id_to_index
)

print("User poster vector exists?", user_poster_vec is not None)

if user_poster_vec is not None:
    print("User poster vector norm:", np.linalg.norm(user_poster_vec))


User poster vector exists? False
